<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/06-tablas-dinamicas-y-buscarv/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 0.6 · pandas II: tablas dinámicas, BUSCARV y estadística básica

**Serie:** Del Excel al Machine Learning · **Temporada 0** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas dos celdas. Abren las cuatro tablas y agregan `anio` y `mes` a `ventas`.

Algunos ejercicios usan tablas de los anteriores, así que **ejecútalos en orden**. Si Colab se desconecta, vuelve a ejecutar todo desde arriba.

La columna `total` de `ventas` suma **S/ 13,582,286.61**. Úsalo para comprobar tus resultados.

In [1]:
import pandas as pd

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
ventas = pd.read_csv(DATOS + "ventas.csv")
productos = pd.read_csv(DATOS + "productos.csv")
clientes = pd.read_csv(DATOS + "clientes.csv")
empleados = pd.read_csv(DATOS + "empleados.csv")

ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="%Y-%m-%d")
ventas["anio"] = ventas["fecha"].dt.year
ventas["mes"] = ventas["fecha"].dt.month

ventas.shape

(50000, 13)

### 1. Las cantidades

Calcula el promedio y la mediana de `cantidad`. Luego muestra su resumen con `.describe()`.

¿Cuál describe mejor una venta típica? ¿Qué te dice el valor máximo? Responde en un comentario.

> 💡 Un comentario es una línea que empieza con `#`.

In [3]:
print(ventas["cantidad"].mean())
print(ventas["cantidad"].median())

ventas["cantidad"].describe()

3.85744
3.0


count    50000.000000
mean         3.857440
std          3.113813
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         41.000000
Name: cantidad, dtype: float64

La mediana describe mejor una venta típica: 3 presentaciones. Las ventas grandes suben el promedio a 3.86. El máximo, 41, está muy lejos de lo normal.

### 2. Lima vende más, pero...

Calcula la venta **promedio** (`total`) por ciudad, ordenada de mayor a menor. El primer lugar debe promediar cerca de S/ 297. Responde en un comentario: ¿Es Lima?

In [4]:
ventas.groupby("ciudad")["total"].mean().sort_values(ascending=False)

# Lima no: es la que más vende en total porque tiene más ventas,
# pero Cusco, Trujillo y Chiclayo tienen ventas promedio más altas.

ciudad
Cusco       296.826301
Trujillo    293.994058
Chiclayo    277.783459
Lima        267.925638
Piura       257.428288
Arequipa    246.716604
Name: total, dtype: float64

### 3. Métodos de pago por año

Crea una tabla con `metodo_pago` en las filas, `anio` en las columnas y la **suma** de `total` en los valores, con los totales generales. Redondéala a 0 decimales.

La esquina de abajo a la derecha debe dar `13582287.0`.

In [5]:
ventas.pivot_table(
    index="metodo_pago",
    columns="anio",
    values="total",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
).round()

anio,2024,2025,Total
metodo_pago,,,
Billetera digital,1688053.0,1730774.0,3418827.0
Crédito,1241411.0,1295036.0,2536446.0
Efectivo,2076137.0,2178750.0,4254887.0
Transferencia,1606443.0,1765683.0,3372126.0
Total,6612044.0,6970243.0,13582287.0


### 4. Cuánto compra cada tipo de cliente

`ventas` no dice si el cliente es bodega, minimarket o restaurante: eso está en `clientes`.

Crea `ventas_clientes`, con `pd.merge()`, trayendo solo `id_cliente` y `tipo`. Comprueba que sigan **50,000 filas** y **0 vacíos** en `tipo`. Luego muestra la suma de `total` por `tipo`, de mayor a menor.

In [6]:
ventas_clientes = pd.merge(ventas, clientes[["id_cliente", "tipo"]], on="id_cliente", how="left")

print(len(ventas_clientes))
print(ventas_clientes["tipo"].isna().sum())

por_tipo = ventas_clientes.groupby("tipo")["total"].sum().sort_values(ascending=False)

print(round(por_tipo.sum(), 2))

por_tipo

50000
0
13582286.61


tipo
Minimarket     6415873.49
Bodega         5107719.70
Restaurante    2058693.42
Name: total, dtype: float64

### 5. ¿Qué compran los restaurantes?

A `ventas_clientes`, agrégale desde `productos` las columnas `categoria` y `costo`, y guarda el resultado en `ventas_completas`. Incluye también la columna común y usa `how="left"`. Comprueba de nuevo que sigan 50,000 filas y que `categoria` no tenga vacíos.

Crea una tabla con `tipo` en las filas, `categoria` en las columnas y la suma de `total`, redondeada a 0 decimales (verás un `.0`, es normal). La celda Bodega / Abarrotes debe dar `1227259.0`. ¿Qué categoría compran más los restaurantes? ¿Y los minimarkets?

In [7]:
columnas = ["id_producto", "categoria", "costo"]

ventas_completas = pd.merge(ventas_clientes, productos[columnas], on="id_producto", how="left")

print(len(ventas_completas))
print(ventas_completas["categoria"].isna().sum())

ventas_completas.pivot_table(
    index="tipo",
    columns="categoria",
    values="total",
    aggfunc="sum",
).round()

50000
0


categoria,Abarrotes,Bebidas,Cuidado personal,Limpieza,Snacks
tipo,,,,,
Bodega,1227259.0,828698.0,1228250.0,1016952.0,806561.0
Minimarket,1498659.0,788555.0,1941329.0,1545223.0,642108.0
Restaurante,1158370.0,250850.0,134089.0,481411.0,33973.0


Los restaurantes compran sobre todo **abarrotes**, y los minimarkets, **cuidado personal**.

### 6. Ranking de vendedores

Crea `ventas_vendedor`: `ventas` con el `nombre` de `empleados`. Incluye también la columna común y usa `how="left"`. Ojo: se llama `id_vendedor` en una tabla e `id_empleado` en la otra. Comprueba que sigan 50,000 filas y que `nombre` no tenga vacíos.

Muestra los **5 vendedores** que más vendieron, de mayor a menor. Quien va primero debe superar los S/ 1.8 millones.

In [8]:
ventas_vendedor = pd.merge(
    ventas,
    empleados[["id_empleado", "nombre"]],
    left_on="id_vendedor",
    right_on="id_empleado",
    how="left",
)

print(len(ventas_vendedor))
print(ventas_vendedor["nombre"].isna().sum())

ventas_vendedor.groupby("nombre")["total"].sum().sort_values(ascending=False).head()

50000
0


nombre
Patricia Chávez Romero    1818773.24
Gloria Rojas Cárdenas     1295117.41
María Espinoza Ramos      1269125.46
Manuel Huanca Mendoza     1137647.14
Karina Castro Ortiz       1093629.39
Name: total, dtype: float64

### 7. ¿Dónde está la ganancia?

La gerencia no solo quiere saber cuánto se vende, sino cuánto se gana. Usa `ventas_completas`, del ejercicio 5.

Crea la columna `ganancia`: `total − (cantidad × costo)`. Es la ganancia bruta, antes de gastos como sueldos y alquileres. Muestra la suma por `categoria`, de mayor a menor, y redondeada a 2 decimales. La suma de toda la columna, también redondeada, debe dar `1488582.79`.

¿Qué porcentaje del total vendido es ganancia? Divide la suma de `ganancia` entre la suma de `total`.

In [9]:
ventas_completas["ganancia"] = (
    ventas_completas["total"] - ventas_completas["cantidad"] * ventas_completas["costo"]
)

print(ventas_completas.groupby("categoria")["ganancia"].sum().sort_values(ascending=False).round(2))

ganancia = ventas_completas["ganancia"].sum()

print(round(ganancia, 2))
print(round(ganancia / ventas_completas["total"].sum() * 100, 1))

categoria
Abarrotes           443448.26
Cuidado personal    352530.03
Limpieza            308277.92
Bebidas             227106.80
Snacks              157219.78
Name: ganancia, dtype: float64
1488582.79
11.0


Cerca del 11 % de lo vendido es ganancia bruta. Abarrotes es la que más soles deja, pero no la de mayor margen: Bebidas gana 12.2 % sobre lo que vende, y Abarrotes, 11.4 %. Como con Lima en el ejercicio 2, el más grande no siempre es el mejor.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 0.7: Gráficos**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0